# Regression Framework — DDL Setup

Run this notebook **once** to create all required Delta tables in `caife_utils`.

**Tables created:**
- `regression_results` — master results table (summary + detail rows)
- `regression_stats` — column profiling and drift detection results

In [ ]:
# ============================================================
# WIDGETS — Override defaults as needed
# ============================================================
dbutils.widgets.text('catalog',       'uc_dev0004',  'Catalog')
dbutils.widgets.text('output_schema', 'caife_utils', 'Output Schema')

CATALOG       = dbutils.widgets.get('catalog')
OUTPUT_SCHEMA = dbutils.widgets.get('output_schema')

print(f'Setting up DDL in: {CATALOG}.{OUTPUT_SCHEMA}')

In [ ]:
# ============================================================
# CREATE SCHEMA IF NOT EXISTS
# ============================================================
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CATALOG}.{OUTPUT_SCHEMA}')
print(f'Schema {CATALOG}.{OUTPUT_SCHEMA} ready.')

In [ ]:
# ============================================================
# TABLE 1: regression_results
# Master table — both SUMMARY and DETAIL rows live here.
#
# row_type = 'SUMMARY' : one row per table per run
#   -> checks_total, checks_passed, checks_failed populated
#   -> column_name, partition_value, expected_value, actual_value = NULL
#
# row_type = 'DETAIL'  : one row per check per table per run
#   -> column_name populated for column-level checks
#   -> partition_value populated for partition checks
#   -> expected_value / actual_value populated where applicable
# ============================================================
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{OUTPUT_SCHEMA}.regression_results (

    -- Run identification
    run_id              STRING    NOT NULL COMMENT 'Unique identifier for the regression run',
    run_timestamp       TIMESTAMP NOT NULL COMMENT 'When the run started',

    -- Row classification
    row_type            STRING    NOT NULL COMMENT 'SUMMARY or DETAIL',

    -- Table identification
    source_table        STRING    NOT NULL COMMENT 'Fully qualified source table: catalog.schema.table',
    target_table        STRING    NOT NULL COMMENT 'Fully qualified target table: catalog.schema.table',

    -- Check details
    check_type          STRING    COMMENT 'row_count | schema | aggregates | column_profile | partition | duplicate | ri | custom_sql | SUMMARY',
    check_status        STRING    NOT NULL COMMENT 'PASSED | FAILED | WARNED | SKIPPED',

    -- Summary counts (populated for SUMMARY rows)
    checks_total        INT       COMMENT 'Total checks run for this table',
    checks_passed       INT       COMMENT 'Total checks passed',
    checks_failed       INT       COMMENT 'Total checks failed',
    checks_warned       INT       COMMENT 'Total checks warned or skipped',

    -- Detail fields (populated for DETAIL rows)
    column_name         STRING    COMMENT 'Column name for column-level checks',
    partition_value     STRING    COMMENT 'Partition value for partition-level checks',
    expected_value      STRING    COMMENT 'Expected value (source)',
    actual_value        STRING    COMMENT 'Actual value (target)',
    failure_reason      STRING    COMMENT 'Detailed reason for failure or warning',

    -- Performance
    duration_seconds    DOUBLE    COMMENT 'Time taken for this check in seconds'
)
USING DELTA
PARTITIONED BY (run_timestamp)
COMMENT 'Regression framework master results table — summary and detail rows per run'
""")

print(f'Table {CATALOG}.{OUTPUT_SCHEMA}.regression_results created.')

In [ ]:
# ============================================================
# TABLE 2: regression_stats
# Column-level profiling stats for both source and target.
# One row per column per table per run per side (source/target).
# ============================================================
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{OUTPUT_SCHEMA}.regression_stats (

    -- Run identification
    run_id              STRING    NOT NULL COMMENT 'Unique identifier for the regression run',
    run_timestamp       TIMESTAMP NOT NULL COMMENT 'When the run started',

    -- Table & column identification
    table_full_path     STRING    NOT NULL COMMENT 'Fully qualified table: catalog.schema.table',
    side                STRING    NOT NULL COMMENT 'SOURCE or TARGET',
    column_name         STRING    NOT NULL COMMENT 'Column name',
    column_type         STRING    COMMENT 'Detected or overridden type: numerical | categorical | date | boolean | complex',

    -- Table-level stats (repeated per column for convenience)
    table_row_count     LONG      COMMENT 'Total rows in table',
    table_size_bytes    LONG      COMMENT 'Table size in bytes',
    table_size_mb       DOUBLE    COMMENT 'Table size in MB',
    table_size_gb       DOUBLE    COMMENT 'Table size in GB',
    table_column_count  INT       COMMENT 'Total columns in table',

    -- Common stats (all column types)
    null_count          LONG      COMMENT 'Number of null values',
    null_pct            DOUBLE    COMMENT 'Percentage of null values',
    distinct_count      LONG      COMMENT 'Distinct value count (approx or exact)',
    is_exact_distinct   BOOLEAN   COMMENT 'True if exact distinct count was used',

    -- Numerical stats
    num_min             DOUBLE    COMMENT 'Minimum value (numerical)',
    num_max             DOUBLE    COMMENT 'Maximum value (numerical)',
    num_sum             DOUBLE    COMMENT 'Sum (numerical)',
    num_mean            DOUBLE    COMMENT 'Mean (numerical)',
    num_stddev          DOUBLE    COMMENT 'Standard deviation (numerical)',
    num_mode            DOUBLE    COMMENT 'Mode (numerical) — only if enable_mode=true',
    num_count           LONG      COMMENT 'Non-null count (numerical)',

    -- Categorical stats
    cat_min_length      INT       COMMENT 'Minimum string length',
    cat_max_length      INT       COMMENT 'Maximum string length',
    cat_avg_length      DOUBLE    COMMENT 'Average string length',
    cat_empty_pct       DOUBLE    COMMENT 'Percentage of empty strings',
    cat_top_n_values    STRING    COMMENT 'Top N frequent values as JSON array',

    -- Date / Timestamp stats
    date_min            STRING    COMMENT 'Minimum date/timestamp as string',
    date_max            STRING    COMMENT 'Maximum date/timestamp as string',
    date_range_days     LONG      COMMENT 'Range in days between min and max',

    -- Boolean stats
    bool_true_pct       DOUBLE    COMMENT 'Percentage of TRUE values',
    bool_false_pct      DOUBLE    COMMENT 'Percentage of FALSE values',

    -- Drift flags (compared to previous run)
    drift_null_pct_flag      BOOLEAN COMMENT 'True if null % increased beyond threshold',
    drift_distinct_pct_flag  BOOLEAN COMMENT 'True if distinct count shifted beyond threshold',
    drift_rowcount_flag      BOOLEAN COMMENT 'True if row count changed beyond threshold',
    drift_mean_flag          BOOLEAN COMMENT 'True if mean shifted beyond threshold (numerical)',
    drift_new_category_flag  BOOLEAN COMMENT 'True if new category appeared vs previous run',
    drift_top_value_changed  BOOLEAN COMMENT 'True if top frequent value changed vs previous run',
    drift_notes         STRING    COMMENT 'Human-readable drift summary'
)
USING DELTA
PARTITIONED BY (run_timestamp)
COMMENT 'Regression framework column profiling and drift detection stats'
""")

print(f'Table {CATALOG}.{OUTPUT_SCHEMA}.regression_stats created.')

In [ ]:
# ============================================================
# VERIFY — Show created tables
# ============================================================
print('\n=== Tables in output schema ===')
spark.sql(f'SHOW TABLES IN {CATALOG}.{OUTPUT_SCHEMA}').show(truncate=False)

print('\n=== regression_results schema ===')
spark.sql(f'DESCRIBE TABLE {CATALOG}.{OUTPUT_SCHEMA}.regression_results').show(100, truncate=False)

print('\n=== regression_stats schema ===')
spark.sql(f'DESCRIBE TABLE {CATALOG}.{OUTPUT_SCHEMA}.regression_stats').show(100, truncate=False)